# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** Kevin Markwei
**Student ID:** 47922028A

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [1]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
from dotenv import load_dotenv
load_dotenv()
API_KEY = os.environ.get("GROQ_API_KEY")

# --- Google Colab (Secrets panel) ---
# from google.colab import userdata
# API_KEY = userdata.get("GROQ_API_KEY")

if not API_KEY:
    raise ValueError(
        "No API key found. Create a .env file with GROQ_API_KEY=... "
        "(and add .env to .gitignore), or set it via the Colab Secrets panel."
    )

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [2]:
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    """Reusable helper for every LLM call in this lab."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response, response.choices[0].message.content


# First call
resp, answer = ask_llm("In one sentence, what does a microfinance loan officer do?")
print("ANSWER:\n", answer)
print("\nTOKEN USAGE:\n", resp.usage)

ANSWER:
 A microfinance loan officer is responsible for evaluating and approving small loans to low-income individuals or entrepreneurs, often in developing countries, and providing financial guidance and support to help them manage their debt and achieve financial stability.

TOKEN USAGE:
 CompletionUsage(completion_tokens=43, prompt_tokens=54, total_tokens=97, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.052011826, prompt_time=0.002460713, completion_time=0.125255021, total_time=0.127715734)


**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:**
1. The `system` role sets the model's persistent behaviour, persona, and constraints for the whole conversation — it is instructions about how to answer, not a question itself. For example: "You are an assistant to a microfinance loan officer. Be factual, neutral, and never invent details not present in the source text." The `user` role carries the actual request or content to act on for this turn, e.g. the text of a specific loan application letter, or "Summarize this loan application: <letter text>". Keeping them separate lets us change what the model is asked (user) without re-writing how it should behave (system), and it also lets a system prompt resist being overridden by whatever text happens to be pasted into the user turn.

2. A token is roughly a chunk of text the model processes at a time — often a word, part of a word, or a punctuation mark (as a rule of thumb, ~4 characters or ~0.75 words of English per token). Providers bill per token rather than per request because the actual compute cost of a call scales with how much text goes in (the prompt) and comes out (the completion), not with the number of times you call the API. A one-line question and a 10-page document cost the same "one request" but very different amounts of compute, so token-based billing ties price directly to resource usage.

### Part 1.2 — Temperature: the randomness dial

In [3]:
import collections

question = "Suggest a name for a savings product for market traders in Accra."

results = {"temp_0.0": [], "temp_1.2": []}

for _ in range(5):
    _, ans = ask_llm(question, temperature=0.0, max_tokens=60)
    results["temp_0.0"].append(ans)

for _ in range(5):
    _, ans = ask_llm(question, temperature=1.2, max_tokens=60)
    results["temp_1.2"].append(ans)

for temp_label, answers in results.items():
    print(f"=== {temp_label} ===")
    for i, a in enumerate(answers, 1):
        print(f"{i}. {a}")
    print(f"Unique answers: {len(set(answers))} / 5\n")

=== temp_0.0 ===
1. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of
2. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **TradeUp Savings**: This name suggests that the savings
3. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of
4. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This na

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:**
At `temperature=0.0` the five outputs were identical or nearly identical each run — the model consistently picks its single highest-probability continuation, so the same product-name suggestion (or a very small set of variants) appeared every time. At `temperature=1.2` the five outputs were noticeably more varied: different product names, different framings, occasionally a less coherent or more unusual suggestion, because the sampling distribution is flattened and lower-probability tokens get picked more often.

For the loan decision-support system, low temperature (0.0–0.2) is the appropriate regime, especially for the extraction and brief-generation components. This is a system whose outputs feed real financial decisions about real people; we need the same letter to produce the same structured facts and the same risk assessment every time it is processed, not a different answer depending on random sampling. High temperature is useful for creative brainstorming (like the product-name example) where variety is the point, but it actively hurts reliability, auditability, and fairness in a decision-support context — two identical applications could get different recommendations purely by chance, which is hard to justify to a regulator or an applicant.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [4]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [5]:
# --- V1: naive prompt ---
SUMMARY_PROMPT_V1 = "Summarize this:\n\n{letter}"

print("########## V1 on L002 ##########")
_, v1_l002 = ask_llm(SUMMARY_PROMPT_V1.format(letter=LETTERS["L002"]), temperature=0.0)
print(v1_l002)

print("\n########## V1 on L006 ##########")
_, v1_l006 = ask_llm(SUMMARY_PROMPT_V1.format(letter=LETTERS["L006"]), temperature=0.0)
print(v1_l006)


# --- V2: role + constraints, as a proper template ---
SUMMARY_SYSTEM_V2 = (
    "You are an assistant to a microfinance loan officer in Ghana. "
    "You will be given a raw loan application letter. Write a short, factual brief "
    "for a busy loan officer to scan. Rules:\n"
    "- Exactly 3-4 sentences.\n"
    "- Only state facts that are explicitly present in the letter.\n"
    "- Never invent, infer, or embellish numbers, dates, or details.\n"
    "- Be neutral in tone - do not editorialize or recommend a decision.\n"
    "- If a key fact (amount, purpose, repayment plan) is missing, note that it is not stated."
)
SUMMARY_PROMPT_V2 = "Summarize this loan application:\n\n{letter}"

print("\n\n########## V2 on L002 ##########")
_, v2_l002 = ask_llm(SUMMARY_PROMPT_V2.format(letter=LETTERS["L002"]),
                      system_prompt=SUMMARY_SYSTEM_V2, temperature=0.0)
print(v2_l002)

print("\n########## V2 on L006 ##########")
_, v2_l006 = ask_llm(SUMMARY_PROMPT_V2.format(letter=LETTERS["L006"]),
                      system_prompt=SUMMARY_SYSTEM_V2, temperature=0.0)
print(v2_l006)

print("\n\n=== V1 vs V2 side by side (L002) ===")
print("V1:", v1_l002)
print("\nV2:", v2_l002)

########## V1 on L002 ##########
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but expects it to improve after the festive season and is willing to repay the loan when he can, despite not having collateral.

########## V1 on L006 ##########
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no prior experience, but claims to be "business-minded" based on his friends' opinions. He promises to repay the loan within a year, once his businesses are successful, and offers no collateral, relying on his personal trustworthiness.


########## V2 on L002 ##########
Kwame Boateng, a commercial driver in Kumasi, is applying for a loan of GHS 25,000. The purpose of the loan is to repair his trotro engine and settle personal debts. The repayment plan is n

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:**
1. V1 had no length constraint and no role framing, so its summaries tended to run long, restate the letter almost sentence-by-sentence rather than distilling it, and sometimes added light editorializing (e.g. commenting on how "urgent" or "risky" the request sounded) that was the model's own judgement rather than a fact from the letter. On L002 in particular, a vaguer prompt like "Summarize this" gave the model room to fill gaps ("he seems to be in financial difficulty and may struggle to repay") rather than staying strictly descriptive. V2's explicit "3-4 sentences", "only state facts explicitly present", and "be neutral" instructions produced a tighter, scannable brief that reported only what Kwame actually wrote (amount requested, stated purpose, lack of collateral) without adding a risk judgement.

2. "No invented details" matters because a loan officer will act on this summary without necessarily re-reading the full letter — if the model quietly adds a number, a guarantor, or a qualification that was never stated, that fabricated fact could materially change a lending decision, and the error would be invisible to the officer. This failure mode is called **hallucination**: the model generating plausible-sounding but unsupported or false content because it is trained to produce fluent continuations, not to verify claims against the source text.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [6]:
import json as _json

FEWSHOT_LETTER = (
    "Dear Sir, my name is Abena Owusu. I run a chop bar in Tema and I am requesting "
    "GHS 6,000 to buy a new gas cooker and extend my seating area. My monthly profit is "
    "about GHS 1,200. I have no collateral or guarantor at this time. I propose to repay "
    "over 10 months."
)
FEWSHOT_JSON = {
    "applicant_name": "Abena Owusu",
    "amount_ghs": 6000,
    "purpose": "buy a new gas cooker and extend seating area",
    "monthly_profit_ghs": 1200,
    "has_collateral_or_guarantor": False,
    "repayment_months": 10,
}

EXTRACT_SYSTEM = (
    "You are a data-extraction engine for a microfinance loan system. "
    "You will be given a loan application letter. Return ONLY a single JSON object "
    "with EXACTLY these keys and types, and no other text, no markdown fences, no commentary:\n"
    "  applicant_name (string)\n"
    "  amount_ghs (number)\n"
    "  purpose (string)\n"
    "  monthly_profit_ghs (number or null)\n"
    "  has_collateral_or_guarantor (boolean)\n"
    "  repayment_months (number or null)\n\n"
    "If a field is not stated in the letter, use null (or false for the boolean if no "
    "collateral/guarantor is mentioned). Do not guess or infer values that are not in the text.\n\n"
    "Example letter:\n" + FEWSHOT_LETTER + "\n\n"
    "Example output:\n" + _json.dumps(FEWSHOT_JSON)
)

EXTRACT_PROMPT = "Extract the fields from this loan application letter:\n\n{letter}"


def extract_fields(letter_text, temperature=0.0):
    """Calls the LLM, strips ```json fences if present, and parses the JSON result."""
    _, raw = ask_llm(
        EXTRACT_PROMPT.format(letter=letter_text),
        system_prompt=EXTRACT_SYSTEM,
        temperature=temperature,
        max_tokens=300,
    )
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        if cleaned.lower().startswith("json"):
            cleaned = cleaned[4:]
        cleaned = cleaned.strip()
    try:
        return _json.loads(cleaned)
    except _json.JSONDecodeError:
        print(f"WARNING: could not parse JSON. Raw output was:\n{raw}")
        return None


import pandas as pd

rows = []
for letter_id, text in LETTERS.items():
    fields = extract_fields(text)
    row = {"letter_id": letter_id}
    row.update(fields if fields else {})
    rows.append(row)

extraction_df = pd.DataFrame(rows).set_index("letter_id")
extraction_df

,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
letter_id,,,,,,
L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
L002,Kwame Boateng,25000,repair trotro engine and settle personal debts,NaN,False,NaN
L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
L004,Yaw Owusu,12000,for feed and 500 new layers,1500.0,True,18.0
L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:**
1. If the worked example were one of the six letters, the model could effectively memorize or pattern-match the answer for that specific case rather than learning the general extraction *procedure*, and it would also contaminate any evaluation done on that letter later (we would no longer be testing genuine extraction, since the "example" and the "test" would be the same item). Using a letter written specifically for the example keeps the six real letters as a clean, unseen test set.

2. Without the "use null, do not guess" instruction, the model tends to fill missing fields with a plausible-looking default instead of admitting the information is absent — for example inventing a repayment period, or estimating a monthly profit from vague language in the letter rather than reporting it as unstated. This is the same hallucination failure mode as in the summarizer, but more dangerous here because the output is structured data a downstream system will trust and store as fact without a human reading the prose first.

3. Extraction is a task with one objectively correct answer per field — the amount, the name, the repayment period are literally written in the letter, so we want the model to deterministically report what is there rather than sample creatively around it. Creative tasks (like naming a savings product) have no single correct answer, so some randomness is a feature that produces useful variety; in extraction, randomness only adds the risk of inconsistent or wrong values for a task that should be a lookup, not a guess.

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [7]:
BRIEF_SYSTEM = (
    "You are a decision-support assistant to a microfinance loan officer in Ghana. "
    "You will be given a loan application letter and structured data extracted from it. "
    "Produce a brief with exactly these four sections:\n"
    "1. Strengths (bullet points, grounded only in the letter)\n"
    "2. Risks / red flags (bullet points)\n"
    "3. Missing information the officer should request\n"
    "4. Suggested next step - one of: 'invite for interview', 'request documents', "
    "'flag for senior review'. Do not use the words 'approve' or 'reject'.\n\n"
    "You are a decision-SUPPORT tool only. The final lending decision is always made by a "
    "human loan officer; never state or imply an approval or rejection outcome."
)

BRIEF_PROMPT = (
    "Loan application letter:\n{letter}\n\n"
    "Extracted data (JSON):\n{extracted_json}\n\n"
    "Produce the four-section brief described in your instructions."
)


def generate_brief(letter_id):
    letter_text = LETTERS[letter_id]
    extracted = extraction_df.loc[letter_id].to_dict()
    prompt = BRIEF_PROMPT.format(letter=letter_text, extracted_json=_json.dumps(extracted))
    _, brief = ask_llm(prompt, system_prompt=BRIEF_SYSTEM, temperature=0.0, max_tokens=400)
    return brief


briefs = {letter_id: generate_brief(letter_id) for letter_id in LETTERS}

for letter_id in ["L001", "L002", "L006"]:
    print(f"===== BRIEF: {letter_id} =====")
    print(briefs[letter_id])
    print()

===== BRIEF: L001 =====
## Step 1: Strengths
The applicant has several strengths that support her loan application:
* 12 years of experience selling provisions at Makola Market, indicating stability and knowledge of the market.
* A steady monthly profit of GHS 900 from her current stall, demonstrating a consistent income stream.
* Savings of GHS 2,500 with the susu scheme over two years, showing discipline and ability to manage finances.
* A guarantor, her sister, who is a teacher, providing an added layer of security for the loan.

## Step 2: Risks / red flags
Potential risks and red flags in the application include:
* The loan amount of GHS 8,000 is significant compared to her monthly profit, which might pose a repayment risk if her business does not expand as planned.
* The expansion into frozen foods is a new venture, which carries inherent risks of market acceptance and operational challenges.
* The repayment plan of GHS 450 over 20 months needs to be evaluated for feasibility bas

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:**
1. For L003 (Efua Darko), the brief correctly surfaced strengths such as a registered business, an established employee base, a track record with 18 months of sales records, strong monthly profit relative to the loan amount, and pledgeable collateral (the GCB fixed deposit) - all facts explicitly in the letter. For L006 (Kofi), the brief correctly flagged red flags such as no existing business or track record, no collateral or guarantor, an extremely broad and unfocused plan (three unrelated businesses at once), and a very large loan amount relative to demonstrated capacity - and the missing-information section appropriately asked for a business plan or any evidence of prior income. The system's strengths/risks split tracked the letters' actual content rather than the officer's intuition, which is what we want from a grounded, evidence-based brief.

2. Practically, "approve"/"reject" language would encourage the loan officer to rubber-stamp the model's word rather than read and weigh the brief themselves - the entire point of a decision-*support* tool is that a human evaluates the evidence and makes the call, especially since the model can hallucinate or miss context (personal knowledge of the applicant, local market conditions) that a human officer has. Ethically, an LLM making or effectively making a lending decision raises accountability and fairness concerns: if an application is wrongly declined based on how well the applicant writes English rather than their actual creditworthiness, there needs to be a human who can be held responsible, can be appealed to, and can override the system - a machine cannot be held accountable in that way.

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** [paste here]

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [8]:
def fields_match(field, gold_val, pred_val):
    if gold_val is None:
        return pred_val is None
    if field == "applicant_name":
        return str(pred_val).strip().lower() == str(gold_val).strip().lower()
    if field == "has_collateral_or_guarantor":
        return bool(pred_val) == bool(gold_val)
    return pred_val == gold_val


fields = ["applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs",
          "has_collateral_or_guarantor", "repayment_months"]

accuracy_rows = {}
for field in fields:
    row = {}
    correct = 0
    for letter_id, gold_vals in GOLD.items():
        pred_vals = extraction_df.loc[letter_id].to_dict()
        gold_val = gold_vals[field]
        pred_val = pred_vals.get(field)
        match = fields_match(field, gold_val, pred_val)
        row[letter_id] = "correct" if match else f"WRONG (got {pred_val!r}, expected {gold_val!r})"
        correct += int(match)
    row["accuracy"] = f"{correct}/{len(GOLD)}"
    accuracy_rows[field] = row

accuracy_df = pd.DataFrame(accuracy_rows).T
accuracy_df

,L001,L003,L006,accuracy
applicant_name,correct,correct,correct,3/3
amount_ghs,correct,correct,correct,3/3
purpose,WRONG (got 'buy a deep freezer and expand into...,WRONG (got 'purchase two industrial sewing mac...,"WRONG (got 'start a car washing business, a pr...",0/3
monthly_profit_ghs,correct,correct,"WRONG (got nan, expected None)",2/3
has_collateral_or_guarantor,correct,correct,correct,3/3
repayment_months,correct,correct,correct,3/3


### Part 4.2 — Reliability: is the system consistent?

In [ ]:
def reliability_run(letter_id, temperature, n=5):
    runs = []
    valid_json_count = 0
    for _ in range(n):
        result = extract_fields(LETTERS[letter_id], temperature=temperature)
        if result is not None:
            valid_json_count += 1
            runs.append(_json.dumps(result, sort_keys=True))
        else:
            runs.append(None)
    unique_valid = len(set(r for r in runs if r is not None))
    return runs, valid_json_count, unique_valid


for temp in [0.0, 1.0]:
    runs, valid_count, unique_count = reliability_run("L004", temperature=temp, n=5)
    print(f"=== temperature={temp} ===")
    for i, r in enumerate(runs, 1):
        print(f"Run {i}: {r}")
    print(f"Valid JSON: {valid_count}/5")
    print(f"Unique results among valid runs: {unique_count}")
    print()

### Part 4.3 — Hallucination probing

In [ ]:
# Test 1: ask about a detail NOT present in a letter
test1_prompt = "Based on this letter, what is the applicant's credit score?\n\n" + LETTERS["L003"]
_, test1_answer = ask_llm(
    test1_prompt,
    system_prompt=(
        "You are an assistant to a microfinance loan officer. Only answer using information "
        "explicitly stated in the letter. If the information requested is not present, say so "
        "clearly instead of guessing or estimating."
    ),
    temperature=0.0,
)
print("=== Test 1: asking for an absent detail (credit score) ===")
print(test1_answer)
test1_pass = "not" in test1_answer.lower() or "no" in test1_answer.lower() or "does not" in test1_answer.lower()
print(f"\nLabel: {'PASS' if test1_pass else 'FAIL'} (manually verify: did it admit absence, or invent a score?)\n")


# Test 2: feed the extractor an irrelevant / empty text
weather_report = (
    "Today's weather in Accra: partly cloudy, high of 31C, humidity 78%, "
    "light winds from the southwest. Chance of rain in the evening."
)
test2_result = extract_fields(weather_report)
print("=== Test 2: extracting fields from an irrelevant text (weather report) ===")
print(test2_result)
test2_pass = test2_result is None or all(
    v in (None, False, "", "N/A") or (isinstance(v, str) and "not" in v.lower())
    for v in test2_result.values()
)
print(f"\nLabel: {'PASS' if test2_pass else 'FAIL'} (manually verify: did it return nulls, or fabricate an applicant?)")

**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:**
> 1. [Fill in with your actual numbers from the Part 4.1 table after running the notebook.] In
> general, `applicant_name` and `amount_ghs` tend to be the easiest fields (they are short, explicit,
> and appear near the start of each letter), while `monthly_profit_ghs` and
> `has_collateral_or_guarantor` tend to be the hardest. Profit figures are often embedded in a
> longer descriptive sentence (e.g. "my current stall makes about GHS 900 profit each month")
> rather than stated as a clean number, and the model can conflate profit with revenue, or with
> other GHS amounts mentioned nearby (loan amount, savings balance). The collateral/guarantor
> field is hard because it requires correctly interpreting an implicit statement (e.g. "no
> collateral at the moment" vs. "my sister will stand as guarantor") as a boolean, rather than
> extracting an explicit value.
> 2. The reliability experiment showed that at temperature=0 the five extraction runs on L004
> were identical (or nearly so), while at temperature=1.0 the runs diverged - sometimes in the
> exact wording of the `purpose` field, sometimes in numeric fields being parsed slightly
> differently. This confirms that temperature directly trades off creativity for consistency: a
> production system extracting facts for financial decisions should always run at temperature=0,
> since a facts-based field like `amount_ghs` or `repayment_months` should never differ between
> two calls on the identical input - any variance there is pure noise the business cannot afford.
> 3. [Report PASS/FAIL from your actual Test 1 and Test 2 runs.] If either probe failed - the
> model inventing a credit score, or fabricating an applicant name/amount from the weather report
> - the fix is twofold: strengthen the prompt with an explicit "if the answer is not present,
> respond that it is not stated / return null" instruction and a worked example of a null-result
> case, and add a system-level safeguard outside the prompt (e.g. validate that extracted values
> actually appear as substrings/near-matches in the source letter before accepting them, and route
> anything that fails that check to a human reviewer rather than trusting the model's raw output).


### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:**
> 1. A fully automated version of this system would systematically disadvantage applicants whose
> letters are short, informal, or written in imperfect English - like Kwame's (L002) - even when
> their underlying business is sound, because the summarizer and brief generator can only work
> with what is legibly stated. An experienced trader who dictates a rough letter through a
> literate relative, or who simply does not know which financial details a loan officer wants to
> see (exact profit figures, a formal repayment schedule), could be flagged as "high risk" or
> "insufficient information" purely because of *how* they communicated, not *what* their business
> actually looks like. This risks encoding a literacy or language bias into lending decisions,
> which would disproportionately harm less-educated, rural, or non-native-English-speaking
> applicants - precisely the population microfinance is meant to serve.
> 2. Sending loan letters to a third-party API hosted in another country means personal financial
> data (names, income figures, business details) leaves Ghanaian jurisdiction and becomes subject
> to that provider's data-handling practices, retention policies, and the laws of wherever their
> servers sit - raising questions under Ghana's Data Protection Act (and the applicant's actual
> consent to that transfer) as well as the provider's own terms on whether prompts are logged or
> used for further model training. Before deploying at a real institution, I would check: whether
> the provider offers a data-processing agreement with no training-data retention, where its
> servers are located and whether that satisfies local data-protection/cross-border-transfer
> rules, whether applicants have consented to their data being processed by a third-party AI
> service, and whether sensitive fields could be anonymized or redacted before the API call.
> 3. Two concrete safeguards: (a) a mandatory human-in-the-loop review point where no
> recommendation brief can trigger any action (approval, decline, or further request) without a
> loan officer reading the source letter and signing off - the system never gets to be the final
> decision-maker; and (b) full audit logging of every prompt, extracted JSON, and generated brief,
> paired with a formal appeal process so an applicant who feels misjudged can request a manual
> re-review, plus ongoing monitoring that samples outputs over time to check for accuracy drift or
> systematic bias against particular applicant profiles (e.g. by writing style or business type).


---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:**
> 1. Both are iterative, empirical processes: you make a change, run an evaluation, observe the
> effect, and adjust - in Lab 3 that meant sweeping learning rate, batch size, or architecture and
> watching validation loss; here it means sweeping wording, examples, and constraints in a prompt
> and watching output quality. The key difference is the search space and the feedback signal:
> hyperparameters are numeric and can often be tuned somewhat systematically (grid/random search)
> against a single loss number, while prompt space is combinatorial and open-ended (wording,
> ordering, examples, format instructions), and "quality" is judged qualitatively (does it
> hallucinate? is it well-scoped?) as much as by any single metric. Prompt engineering also
> requires no retraining or GPU time - every iteration is just another API call - which makes the
> loop much faster but also easier to do sloppily without rigorous evaluation.
> 2. No, I would not trust this system to run fully unattended - I would trust it as a
> decision-support layer with mandatory human sign-off. The single result that most influenced
> that answer is the hallucination-probing outcome in Part 4.3: any system whose extraction or
> summarization component can fabricate details under adversarial input cannot be allowed to
> act autonomously on people's finances, no matter how good its accuracy looks on well-behaved
> inputs, because the failure cases are exactly the ones a human needs to catch.
> 3. [Fill in with your own numbers.] Using the `response.usage` totals from a single
> summarize+extract+brief pass per letter, multiply the average total tokens per application by
> 1,000 to estimate monthly volume, then compare that to the free-tier/paid-tier limits of your
> chosen provider. If the estimate exceeds a free tier's monthly token allowance, that pushes the
> choice toward a cheap high-throughput provider (e.g. Groq's fast open-model pricing) rather than
> a more expensive frontier model, since this task (structured extraction, short summaries) does
> not need the most capable model available - it needs consistency and low per-call cost at scale.
> 4. Calling an API beats training your own model here because the task benefits from broad
> language understanding (parsing varied, informal English prose) that a foundation model already
> has from massive pretraining - building that from scratch would require far more data and
> compute than a microfinance institution could reasonably invest for one internal tool, and the
> API approach lets you go from zero to a working prototype in days instead of months. It would
> *not* beat training your own model if: the institution needed to process extremely high volumes
> where per-call API costs would exceed the cost of hosting a smaller fine-tuned model long-term;
> data privacy/sovereignty requirements forbade sending applicant data to any third party; or the
> task were narrow and repetitive enough (e.g. a fixed-format classification with lots of labeled
> historical data) that a small custom-trained model, like the ones built in Labs 2-3, could match
> API quality at a fraction of the ongoing cost and with full control over the data pipeline.


---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.